In [ ]:
from pipeline_score import pipeline_score
from utils.logging_utils import setup_logging
from pathlib import Path

setup_logging(level="INFO", log_file="logs/run.log", with_emoji=True, enable_prints=True)
configs_yaml = 'config/default_config.yaml'
path_configs = Path(configs_yaml)
df_score, df_sem_score = pipeline_score(path_configs)

2025-11-11 17:41:49,507 INFO [utils.logging_utils] ✨ Logging pronto | level=INFO | file=logs/run.log
2025-11-11 17:41:49,509 INFO [utils.logging_utils] ✨ Logging pronto | level=INFO | file=logs/pipeline.log
2025-11-11 17:41:49,510 INFO [step] ⚙️ Lendo YAML de configuração
2025-11-11 17:41:49,513 INFO [step] ✅ Concluído em 0.00s: Lendo YAML de configuração
2025-11-11 17:41:49,513 INFO [step] 🗃️ Conectando ao Oracle e executando SQL
c:\Users\Kauê Baesso\Downloads\Simplificando um Sistema de Scoring de Clientes em Python\pipeline_score.py:33: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_vendas = pd.read_sql(sql_vendas, conexao)


⚙️ Iniciando: Lendo YAML de configuração
⚙️ Métricas configuradas: 4
✅ Concluído: Lendo YAML de configuração (0.00s)
🗃️ Iniciando: Conectando ao Oracle e executando SQL


# Testar api score

In [ ]:
import requests
import json
from datetime import datetime

class ScoreAPIClient:
    def __init__(self, base_url="http://tomcat03.gam.com.br:8081"):
        self.base_url = base_url
        self.token = None
        self.headers = {'Content-Type': 'application/json'}
    
    def login(self, username, password):
        """
        Faz login na API e obtém o token JWT
        """
        try:
            url = f"{self.base_url}/auth"
            data = {
                "login": username,
                "senha": password
            }
            
            print(f"Fazendo login para usuário: {username}")
            response = requests.post(url, json=data, headers=self.headers)
            
            if response.status_code == 200:
                result = response.json()
                self.token = result.get('token')
                
                # Adiciona o token no header para próximas requisições
                self.headers['Authorization'] = f'Bearer {self.token}'
                
                print("Login realizado com sucesso!")
                print(f"Token obtido: {self.token[:50]}...")
                return True
            else:
                print(f"Erro no login: {response.status_code}")
                print(response.text)
                return False
                
        except requests.exceptions.RequestException as e:
            print(f"Erro de conexão no login: {e}")
            return False
    
    def cadastrar_categoria(self, descricao):
        """
        Cadastra uma nova categoria de score
        """
        try:
            url = f"{self.base_url}/score/categoria/cadastrar"
            data = {"dsCategoria": descricao}
            
            print(f"Cadastrando categoria: {descricao}")
            response = requests.post(url, json=data, headers=self.headers)
            
            if response.status_code == 200:
                result = response.json()
                if result['status'] == 'OK':
                    categoria_id = result['conteudo']['cdCategoria']
                    print(f"Categoria cadastrada! ID: {categoria_id}")
                    return categoria_id
                else:
                    print(f"Erro: {result['mensagem']}")
                    return None
            else:
                print(f"Erro HTTP: {response.status_code}")
                return None
                
        except requests.exceptions.RequestException as e:
            print(f"Erro de conexão: {e}")
            return None
    
    def cadastrar_indicador(self, descricao):
        """
        Cadastra um novo indicador de score
        """
        try:
            url = f"{self.base_url}/score/indicador/cadastrar"
            data = {"dsIndicador": descricao}
            
            print(f"Cadastrando indicador: {descricao}")
            response = requests.post(url, json=data, headers=self.headers)
            
            if response.status_code == 200:
                result = response.json()
                if result['status'] == 'OK':
                    indicador_id = result['conteudo']['cdIndicador']
                    print(f"Indicador cadastrado! ID: {indicador_id}")
                    return indicador_id
                else:
                    print(f"Erro: {result['mensagem']}")
                    return None
            else:
                print(f"Erro HTTP: {response.status_code}")
                return None
                
        except requests.exceptions.RequestException as e:
            print(f"Erro de conexão: {e}")
            return None
    
    def cadastrar_peso(self, categoria_id, indicador_id, peso):
        """
        Cadastra um peso relacionando categoria e indicador
        """
        try:
            url = f"{self.base_url}/score/peso/cadastrar"
            data = {
                "cdCategoria": categoria_id,
                "cdIndicador": indicador_id,
                "pcPeso": peso
            }
            
            print(f"Cadastrando peso: Categoria {categoria_id} + Indicador {indicador_id} = {peso}%")
            response = requests.post(url, json=data, headers=self.headers)
            
            if response.status_code == 200:
                result = response.json()
                if result['status'] == 'OK':
                    print("Peso cadastrado com sucesso!")
                    return True
                else:
                    print(f"Erro: {result['mensagem']}")
                    return False
            else:
                print(f"Erro HTTP: {response.status_code}")
                return False
                
        except requests.exceptions.RequestException as e:
            print(f"Erro de conexão: {e}")
            return False
    
    def cadastrar_score(self, cliente_id, score):
        """
        Cadastra ou atualiza o score de um cliente
        """
        try:
            url = f"{self.base_url}/score/cadastrar"
            data = {
                "cdCliente": cliente_id,
                "dcScore": score
            }
            
            print(f"Cadastrando score: Cliente {cliente_id} = {score}")
            response = requests.post(url, json=data, headers=self.headers)
            
            if response.status_code == 200:
                result = response.json()
                if result['status'] == 'OK':
                    print("Score cadastrado com sucesso!")
                    return True
                else:
                    print(f"Erro: {result['mensagem']}")
                    return False
            else:
                print(f"Erro HTTP: {response.status_code}")
                return False
                
        except requests.exceptions.RequestException as e:
            print(f"Erro de conexão: {e}")
            return False
    
    def consultar_score(self, cliente_id):
        """
        Consulta o score de um cliente
        """
        try:
            url = f"{self.base_url}/score/{cliente_id}"
            
            print(f"Consultando score do cliente: {cliente_id}")
            response = requests.get(url, headers=self.headers)
            
            if response.status_code == 200:
                result = response.json()
                if result['status'] == 'OK':
                    score = result['conteudo']['dcScore']
                    print(f"Score encontrado: {score}")
                    return score
                else:
                    print(f"Erro: {result['mensagem']}")
                    return None
            else:
                print(f"Erro HTTP: {response.status_code}")
                return None
                
        except requests.exceptions.RequestException as e:
            print(f"Erro de conexão: {e}")
            return None
    
    def listar_categorias(self):
        """
        Lista todas as categorias
        """
        try:
            url = f"{self.base_url}/score/categoria/listar"
            
            print("Listando categorias...")
            response = requests.get(url, headers=self.headers)
            
            if response.status_code == 200:
                result = response.json()
                if result['status'] == 'OK':
                    categorias = result['conteudo']
                    print(f"{len(categorias)} categorias encontradas:")
                    for cat in categorias:
                        print(f"   - ID: {cat['cdCategoria']} | {cat['dsCategoria']}")
                    return categorias
                else:
                    print(f"Erro: {result['mensagem']}")
                    return []
            else:
                print(f"Erro HTTP: {response.status_code}")
                return []
                
        except requests.exceptions.RequestException as e:
            print(f"Erro de conexão: {e}")
            return []
    
    def listar_indicadores(self):
        """
        Lista todos os indicadores
        """
        try:
            url = f"{self.base_url}/score/indicador/listar"
            
            print("Listando indicadores...")
            response = requests.get(url, headers=self.headers)
            
            if response.status_code == 200:
                result = response.json()
                if result['status'] == 'OK':
                    indicadores = result['conteudo']
                    print(f"{len(indicadores)} indicadores encontrados:")
                    for ind in indicadores:
                        print(f"   - ID: {ind['cdIndicador']} | {ind['dsIndicador']}")
                    return indicadores
                else:
                    print(f"Erro: {result['mensagem']}")
                    return []
            else:
                print(f"Erro HTTP: {response.status_code}")
                return []
                
        except requests.exceptions.RequestException as e:
            print(f"Erro de conexão: {e}")
            return []
    
    def listar_categorias_indicadores_pesos(self):
        """
        Lista todas as categorias, indicadores e pesos configurados
        """
        try:
            url = f"{self.base_url}/score/categorias-indicadores-pesos"
            
            print("Listando configuração completa...")
            response = requests.get(url, headers=self.headers)
            
            if response.status_code == 200:
                result = response.json()
                if result['status'] == 'OK':
                    dados = result['conteudo']
                    print(f"{len(dados)} configurações encontradas:")
                    for item in dados:
                        print(f"   - Categoria: {item['dsCategoria']} | Indicador: {item['dsIndicador']} | Peso: {item['pcPeso']}%")
                    return dados
                else:
                    print(f"Erro: {result['mensagem']}")
                    return []
            else:
                print(f"Erro HTTP: {response.status_code}")
                return []
                
        except requests.exceptions.RequestException as e:
            print(f"Erro de conexão: {e}")
            return []



In [ ]:
def exemplo_simples():
    """
    Exemplo simples focado apenas em scores
    """
    print("Exemplo simples - apenas scores")
    print("=" * 40)
    
    client = ScoreAPIClient()
    
    # Login
    if client.login("seu_usuario", "sua_senha"):
        # Cadastrar alguns scores
        client.cadastrar_score(16684, 8)
        client.cadastrar_score(12345, 7)
        
        # Consultar os scores
        client.consultar_score(16684)
        client.consultar_score(12345)
    
    print("Exemplo simples finalizado!")


if __name__ == "__main__":
    print("CLIENTE PYTHON PARA API DE SCORE")
    print("Escolha o exemplo:")
    print("1 - Exemplo completo (todas as funcionalidades)")
    print("2 - Exemplo simples (apenas scores)")
    
    escolha = input("Digite sua escolha (1 ou 2): ").strip()
    
    if escolha == "1":
        exemplo_completo()
    elif escolha == "2":
        exemplo_simples()
    else:
        print("Opção inválida. Executando exemplo simples...")
        exemplo_simples()

In [8]:

"""
Exemplo completo de uso da API
"""
print("Iniciando exemplo completo da API de Score")
print("=" * 60)

# Criar cliente da API
client = ScoreAPIClient()

# 1. FAZER LOGIN
print("\nETAPA: AUTENTICAÇÃO")
client.login("api.registro.sac", "#api.R3g1str0.S@C!")

Iniciando exemplo completo da API de Score

ETAPA: AUTENTICAÇÃO
Fazendo login para usuário: api.registro.sac
Login realizado com sucesso!
Token obtido: eyJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJBcGkgU2VydmnDp29zI...


True

In [9]:
# 2. CADASTRAR CATEGORIAS
print("\nETAPA: CADASTRAR CATEGORIAS")
cat1_id = client.cadastrar_categoria("Valor Comercial")
cat2_id = client.cadastrar_categoria("Risco")


ETAPA: CADASTRAR CATEGORIAS
Cadastrando categoria: Valor Comercial
Categoria cadastrada! ID: 3
Cadastrando categoria: Risco
Categoria cadastrada! ID: 4


In [ ]:

# 3. CADASTRAR INDICADORES
print("\nETAPA: CADASTRAR INDICADORES")  
ind1_id = client.cadastrar_indicador("LTV (Faturamento 3 Meses)")
ind2_id = client.cadastrar_indicador("Ticket Medio")
ind3_id = client.cadastrar_indicador("Constância")
ind4_id = client.cadastrar_indicador("Inadimplencia")

# 4. CADASTRAR PESOS (se conseguiu criar categorias e indicadores)
if all([cat1_id, cat2_id, ind1_id, ind2_id, ind3_id, ind4_id]):
    print("\nETAPA: CADASTRAR PESOS")
    client.cadastrar_peso(cat1_id, ind1_id, 35)  
    client.cadastrar_peso(cat1_id, ind2_id, 10)  
    client.cadastrar_peso(cat1_id, ind3_id, 15)  
    client.cadastrar_peso(cat2_id, ind4_id, 40)  



In [ ]:
# 5. CADASTRAR SCORES DE CLIENTES
print("\nETAPA: CADASTRAR SCORES")
clientes_scores = [
    (16684, 8),
    (12345, 7),
    (98765, 9),
    (55555, 6)
]


In [34]:
df_score

,CD_CLIENTE,CD_SISTEMA_NEGOCIO,FREQUENCIA_PEDIDOS,FREQUENCIA_DIAS,VL_FATURADO_BRUTO,TICKET_MEDIO_PEDIDO,CONSISTENCIA_DESVIO,NUM_PEDIDOS,INAD_FLAG,SCORE__LTV,SCORE__Consistencia,SCORE__TicketMedio,SCORE__Inadimplencia,SCORE_TOTAL
0,42179,1,50,18,1.022940e+06,20458.797600,0.891189,50,False,34.976962,12.326433,9.748107,40.0,97.051502
1,24427,1,40,18,4.859795e+05,12149.487500,0.998650,40,False,34.879052,12.004051,9.474811,40.0,96.357914
2,37343,1,48,17,6.874691e+05,14322.272708,1.056236,48,False,34.942406,11.831293,9.553836,40.0,96.327535
3,63789,1,8,3,3.275632e+05,40945.400000,1.133893,8,False,34.758104,11.598320,9.914389,40.0,96.270813
4,14923,1,248,21,3.659860e+05,1475.750194,0.445510,248,False,34.809939,13.663469,7.716497,40.0,96.189905
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18403,73268,5,40,15,1.754558e+04,438.639500,1.375514,40,False,7.954545,10.873459,4.545455,40.0,63.373459
18404,49147,5,43,15,1.727521e+04,401.749086,1.408453,43,False,6.363636,10.774641,2.727273,40.0,59.865550
18405,37897,5,50,16,1.678571e+04,335.714231,1.118034,50,False,4.772727,11.645898,0.454545,40.0,56.873171
18406,34690,5,32,13,1.397037e+04,436.574062,1.640220,32,False,1.590909,10.079339,4.090909,40.0,55.761157


In [ ]:
for i, row in df_score.iterrows():
    cd_cliente = row['CD_CLIENTE']
    score = row['SCORE_TOTAL']

    client.cadastrar_score(cd_cliente, score)

In [ ]:

for cliente_id, score in clientes_scores:
    client.cadastrar_score(cliente_id, score)

# 6. CONSULTAR SCORES
print("\nETAPA: CONSULTAR SCORES")
for cliente_id, _ in clientes_scores:
    client.consultar_score(cliente_id)

# 7. LISTAR DADOS CADASTRADOS
print("\nETAPA: LISTAR DADOS")
client.listar_categorias()
print()
client.listar_indicadores()
print()
client.listar_categorias_indicadores_pesos()

print("\nExemplo completo finalizado!")
print("=" * 60)